# ⚙️ Data Preprocessing — MInDS-14 en-US

Notebook ini melakukan tahap **preprocessing** dataset MInDS-14 en-US sebelum digunakan untuk training/fine-tuning model speech.

Pipeline preprocessing yang dilakukan:
1. **Load dataset** dari Hugging Face
2. **Cek struktur data** — kolom, jumlah sampel, contoh data
3. **Hapus kolom** yang tidak relevan untuk training
4. **Split stratified** → Train / Validation / Test
5. **Resample** audio ke 16.000 Hz
6. **Augmentasi data train** untuk memperbanyak variasi sampel
7. **Simpan dataset** ke disk (Google Drive)

> 🔗 Dataset: [https://huggingface.co/datasets/PolyAI/minds14](https://huggingface.co/datasets/PolyAI/minds14)

In [ ]:
!pip install datasets audiomentations -q
from datasets import load_dataset, DatasetDict, Dataset
from datasets.features import Audio
from sklearn.model_selection import train_test_split
import pandas as pd
import audiomentations as A
from tqdm import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 9.5 MB/s eta 0:00:00


---
## 📥 Load Dataset

Dataset MInDS-14 subset **en-US** di-load langsung dari Hugging Face menggunakan `load_dataset`.

Dataset ini hanya tersedia dalam satu split bawaan (`train`) dengan total **563 sampel**, sehingga split manual ke train/val/test akan dilakukan di tahap selanjutnya.

In [ ]:
dataset = load_dataset("PolyAI/minds14", "en-US")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en-US/train-00000-of-00001.parquet:   0%|          | 0.00/34.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/563 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['path', 'audio', 'transcription', 'english_transcription', 'intent_class', 'lang_id'],
        num_rows: 563
    })
})

---
## 🔎 Cek Struktur Data

Sebelum preprocessing, dilakukan inspeksi awal untuk memastikan:
- Kolom apa saja yang tersedia
- Jumlah total sampel
- Tampilan contoh satu baris data

Hal ini penting untuk memahami data mentah sebelum memutuskan kolom mana yang akan dipertahankan dan mana yang dihapus.

In [ ]:
print("\n-------Struktur Data-----------")
print(f"Kolom: {dataset['train'].column_names}")
print(f"Jumlah Data Train Asli: {len(dataset['train'])}")
print(f"\n Contoh Data:")
print( dataset['train'][0])


-------Struktur Data-----------
Kolom: ['path', 'audio', 'transcription', 'english_transcription', 'intent_class', 'lang_id']
Jumlah Data Train Asli: 563

 Contoh Data:
{'path': 'en-US~JOINT_ACCOUNT/602ba55abb1e6d0fbce92065.wav', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7a0d340c5340>, 'transcription': 'I would like to set up a joint account with my partner', 'english_transcription': 'I would like to set up a joint account with my partner', 'intent_class': 11, 'lang_id': 4}


---
## 🗑️ Hapus Kolom yang Tidak Digunakan

Dari semua kolom yang tersedia di dataset, hanya **3 kolom** yang dibutuhkan untuk training model:
- `audio` — input utama model (sinyal waveform)
- `transcription` — label teks (opsional, berguna jika melatih ASR)
- `intent_class` — label target untuk klasifikasi intent

Kolom yang dihapus:

| Kolom | Alasan Dihapus |
|-------|----------------|
| `lang_id` | Semua sampel adalah en-US, tidak variatif |
| `english_transcription` | Duplikasi informasi dari `transcription` |
| `path` | Hanya path file lokal, tidak dibutuhkan setelah audio di-load ke memori |

Menghapus kolom yang tidak perlu mengurangi ukuran dataset di memori dan menyederhanakan pipeline.

In [ ]:
columns_to_remove = ['lang_id', 'english_transcription', 'path']
dataset = dataset.remove_columns(columns_to_remove)
dataset

DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 563
    })
})

---
## ✂️ Split Data — Stratified (80 : 10 : 10)

Dataset dibagi menjadi tiga subset menggunakan **stratified split** berdasarkan `intent_class`.

| Split | Proporsi | Fungsi |
|-------|----------|--------|
| Train | 80% (~450) | Data untuk melatih model |
| Validation | 10% (~56) | Monitoring & tuning saat training |
| Test | 10% (~57) | Evaluasi akhir yang tidak terlihat selama training |

**Kenapa stratified?**
Karena distribusi intent class di dataset sedikit imbalanced, stratified split memastikan proporsi setiap kelas terjaga di ketiga subset — sehingga tidak ada kelas yang hilang atau sangat sedikit di val/test.

`random_state=42` digunakan agar split **reproducible**.

In [ ]:
full_data = dataset['train']
indices=list(range(len(full_data)))
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=full_data['intent_class'])
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42, stratify=[full_data['intent_class'][i] for i in temp_idx])

#Buat Dataset Splits
train_original = full_data.select(train_idx)
val_original = full_data.select(val_idx)
test_original = full_data.select(test_idx)

---
## 🔄 Resample & Augmentasi Data Train

### Resample ke 16.000 Hz
Dataset MInDS-14 memiliki native sampling rate **8.000 Hz**. Model speech modern seperti Whisper dan Wav2Vec2 mensyaratkan input audio pada **16.000 Hz**.

Resampling dilakukan dengan `cast_column("audio", Audio(sampling_rate=16_000))` yang secara otomatis menginterpolasi sinyal dari 8kHz → 16kHz.

In [ ]:
#Resample
train_original_resampled = train_original.cast_column("audio", Audio(sampling_rate=16_000))

In [ ]:
#Augment

#Definisi
augmenter = A.Compose([
    A.AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
    A.PitchShift(min_semitones=-2, max_semitones=2, p=0.5),
    A.TimeStretch(min_rate=0.8, max_rate=1.25, p=0.5),
])

# Initialize empty lists to store augmented data
augmented_audios_data = [] # This will store dicts: {'array': ..., 'sampling_rate': ...}
augmented_intents = []
augmented_transcripts = []

for sample in tqdm(train_original_resampled):
  audio_array = sample['audio']['array']
  sr = sample['audio']['sampling_rate']
  intent = sample['intent_class']
  transcript = sample['transcription']

  # Add original sample
  augmented_audios_data.append({'array': audio_array, 'sampling_rate': sr})
  augmented_intents.append(intent)
  augmented_transcripts.append(transcript)

  # Add 3 augmented versions per original sample
  for _ in range(3):
    augmented_audio_array = augmenter(samples=audio_array, sample_rate=sr)
    augmented_audios_data.append({'array': augmented_audio_array, 'sampling_rate': sr})
    augmented_intents.append(intent)
    augmented_transcripts.append(transcript)

# Create the new dataset from the augmented data
train_augmented = Dataset.from_dict({
    'audio': augmented_audios_data,
    'intent_class': augmented_intents,
    'transcription': augmented_transcripts
}, features=train_original_resampled.features)

print(f"Selesai! Jumlah data train sebelum augmentasi: {len(train_original_resampled) // 4} -> setelah augmentasi: {len(train_augmented)}")

100%|██████████| 450/450 [01:34<00:00,  4.75it/s]


Selesai! Jumlah data train sebelum augmentasi: 112 -> setelah augmentasi: 1800


---
## 🔄 Resample Val & Test (Tanpa Augmentasi)

Data validasi dan test juga di-resample ke **16.000 Hz** agar konsisten dengan data train.

> ⚠️ **Augmentasi hanya diterapkan pada data train**, bukan pada val dan test.
> Tujuan val dan test adalah mengukur performa model pada data yang representatif terhadap kondisi nyata — jika di-augmentasi, hasil evaluasi tidak akan mencerminkan performa sebenarnya.

In [ ]:
val_resampled = val_original.cast_column("audio", Audio(sampling_rate=16000))
test_resampled = test_original.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
#Gabungkan ke DatasetDict
dataset_split_minds14 = DatasetDict({
    'train': train_augmented,
    'validation': val_resampled,
    'test': test_resampled
})

dataset_split_minds14

DatasetDict({
    train: Dataset({
        features: ['audio', 'intent_class', 'transcription'],
        num_rows: 1800
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 56
    })
    test: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 57
    })
})

---
## 💾 Simpan Dataset ke Disk

Dataset yang sudah diproses disimpan ke **Google Drive** menggunakan format HuggingFace `save_to_disk`.

Format ini lebih efisien dibandingkan menyimpan sebagai CSV/JSON karena:
- Mempertahankan tipe data asli (termasuk array audio)
- Dapat di-load kembali dengan cepat menggunakan `load_from_disk`
- Mendukung lazy loading untuk dataset besar

In [ ]:
save_dir = "/content/drive/MyDrive/Colab Notebooks/data project 3 (Voice Recognition)/minds14_augmented"
dataset_split_minds14.save_to_disk(save_dir)

Saving the dataset (0/2 shards):   0%|          | 0/1800 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/56 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/57 [00:00<?, ? examples/s]

### 📄 Simpan Metadata

File `metadata.json` disimpan bersama dataset sebagai **dokumentasi preprocessing** yang mencatat:
- Jumlah sampel setiap split
- Faktor augmentasi yang digunakan
- Metode augmentasi yang diterapkan
- Sampling rate hasil akhir
- Tanggal pembuatan dataset

Metadata ini penting untuk **reproducibility** dan referensi saat melakukan eksperimen model.

In [ ]:
import json
from datetime import datetime

metadata = {
    "dataset_name": "minds14_enUS_augmented",
    "created_at": datetime.now().isoformat(),
    "original_samples": len(train_original),
    "augmented_train_samples": len(train_augmented),
    "validation_samples": len(val_resampled),
    "test_samples": len(test_resampled),
    "augmentation_factor": 4,  # 1 original + 3 augmented
    "sampling_rate": 16000,
    "split_seed": 42,
    "columns": ['audio', 'transcription', 'intent_class'],
    "augmentation_applied_only_to_train": True,
    "augmentation_methods": ["AddGaussianNoise", "PitchShift", "TimeStretch"]
}

with open(f"{save_dir}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

In [ ]:
# Verifikasi Save
from datasets import load_from_disk

print("\n🔍 Verifying saved dataset...")
loaded_back = load_from_disk(save_dir)
print(f"✓ Load successful!")
print(f"  Train: {len(loaded_back['train'])} samples")
print(f"  Validation: {len(loaded_back['validation'])} samples")
print(f"  Test: {len(loaded_back['test'])} samples")


🔍 Verifying saved dataset...
✓ Load successful!
  Train: 1800 samples
  Validation: 56 samples
  Test: 57 samples


In [ ]:
#Zip Untuk Download
import shutil
zip_path = f"{save_dir}.zip"
print(f"\n Creating zip archive: {zip_path}")
shutil.make_archive(save_dir, 'zip', save_dir)
print(f" Zip created: {zip_path}")

print("\n All done! Dataset ready to use.")


📦 Creating zip archive: /content/drive/MyDrive/Colab Notebooks/data project 3 (Voice Recognition)/minds14_augmented.zip
✅ Zip created: /content/drive/MyDrive/Colab Notebooks/data project 3 (Voice Recognition)/minds14_augmented.zip

🎉 All done! Dataset ready to use.
